# mT5-small 训练示例

这个 notebook 演示如何加载 `prompt -> output` 数据集，并使用 `mT5-small` 进行微调。

In [ ]:
# 如果环境中还没有安装依赖，可以先执行这一格
# 在 Kaggle 中通常只需要补装 transformers、datasets、sentencepiece

# !pip install transformers datasets sentencepiece accelerate

In [ ]:
# 导入基础库
import json
from pathlib import Path

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

print("依赖导入完成")

In [ ]:
# 配置路径和模型名称
# 在 Kaggle 上时，把 data_path 改成你上传后的实际路径

model_name = "google/mt5-small"
data_path = Path("datasets/eda_train_50000.jsonl")
output_dir = "mt5_eda_output"

print("模型:", model_name)
print("数据路径:", data_path)
print("输出目录:", output_dir)

In [ ]:
# 读取 JSONL 数据
# 这里把每条样本转换成 input_text 和 target_text
# target_text 先直接用 JSON 字符串表示，方便 seq2seq 模型学习

records = []
with data_path.open("r", encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        item = json.loads(line)
        records.append({
            "input_text": item["prompt"],
            "target_text": json.dumps(item["output"], ensure_ascii=False, separators=(",", ":"))
        })

print("样本总数:", len(records))
print("第一条输入:")
print(records[0]["input_text"])
print()
print("第一条输出:")
print(records[0]["target_text"][:500])

In [ ]:
# 构建 Hugging Face Dataset，并切分训练集和验证集
# 这里先按 95% / 5% 切分

dataset = Dataset.from_list(records)
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)

dataset_dict = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"]
})

print(dataset_dict)

In [ ]:
# 加载 tokenizer 和模型
# mT5 是 encoder-decoder 架构，适合做文本到结构化输出

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("模型加载完成")

In [ ]:
# 定义预处理函数
# 这里分别对输入文本和目标文本做 tokenize
# max_length 可以后面根据显存和输出长度再微调

max_input_length = 192
max_target_length = 512

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=max_input_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=max_target_length,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# 对数据集进行 tokenize
# remove_columns 用于去掉原始字符串列，保留训练需要的字段

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
)

print(tokenized_datasets)

In [ ]:
# 构建数据整理器
# 它会在 batch 组装时自动补齐长度，并处理 labels 的 padding

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("data collator 已准备好")

In [ ]:
# 配置训练参数
# 这里是一个偏保守、容易先跑通的配置
# 在 Kaggle 上如果显存不足，可以把 batch size 再往下调

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=False,
    report_to="none"
)

print(training_args)

In [ ]:
# 构建 Trainer
# 第一版先不加复杂指标，先让训练流程稳定跑通

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("trainer 构建完成")

In [ ]:
# 开始训练
# 真正训练时再取消注释

# trainer.train()

In [ ]:
# 训练完成后可以保存模型和 tokenizer

# trainer.save_model(output_dir)
# tokenizer.save_pretrained(output_dir)

In [ ]:
# 这里给一个最简单的推理示例
# 注意：只有训练完成后，这里的结果才会接近你的目标输出

test_prompt = "设计一个完成RC低通滤波功能的 EDA 原理图，包含输入 VIN、电阻 R1、输出 OUT、电容 C1 和 接地 GND。请给出所有节点的坐标以及节点之间的连接关系。"

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=max_input_length)
generated_ids = model.generate(**inputs, max_length=max_target_length)
result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(result)